Impact of Fixed Mortgage Rates on Housing Sale Prices

Goal: Examine the correlation between macroeconomic factors (interest rates) and real estate market valuations. Significance: Understanding this relationship helps buyers and investors predict market shifts based on central bank policies. Hypothesis: Higher mortgage rates increase the cost of borrowing, leading to a decrease in housing demand and, consequently, lower sale prices. Data Sourcing & Consolidation: Two independent datasets

Housing Data (Kaggle)
Macroeconomic Data (FRED)


Loading libraries

In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error

%matplotlib inline
sns.set(style="whitegrid")


Loading the two independent datasets

In [ ]:
train_df = pd.read_csv(r"C:\Users\user\Desktop\Datasets\train.csv", encoding="latin")
mortgage_df = pd.read_csv(r"C:\Users\user\Desktop\Datasets\MORTGAGE30US.csv", encoding="latin")

Merging the two datasets into one

In [ ]:
# Processing dates for merging
mortgage_df['observation_date'] = pd.to_datetime(mortgage_df['observation_date'])
mortgage_df['YrSold'] = mortgage_df['observation_date'].dt.year
mortgage_df['MoSold'] = mortgage_df['observation_date'].dt.month

# Average rates per month
monthly_rates = mortgage_df.groupby(['YrSold', 'MoSold'])['MORTGAGE30US'].mean().reset_index()

# Merging Kaggle data with FRED data
df = pd.merge(train_df, monthly_rates, on=['YrSold', 'MoSold'], how='left')
print(f"Combined dataset shape: {df.shape}")


Mathematical Foundation:
Mathematical Theory: We use Multiple Linear Regression to model the price

ln(1 + SalePrice) = b0 + b1(MortgageRate) + b2(GrLivArea) + b3(OverallQual) + epsilon

ln(1 + SalePrice): The log-transformed target variable (Log-Price). Using ln(1+x) ensures the model handles any zero values gracefully and shrinks the impact of extreme outliers.
b0: The intercept, representing the baseline price when all predictors are zero.
b1 (Mortgage Rate): Our primary macroeconomic predictor. We expect b1 < 0, meaning that as interest rates rise, the predicted sale price decreases (inverse relationship).
b2, b3: Control variables (Living Area and Overall Quality) that account for the intrinsic value of the property.
epsilon: The error term, representing the white noise and factors not captured by our model.



Exploratory Data Analysis (EDA) and Transformation

In [ ]:
# Apply log transformation to handle price skewness
df['LogSalePrice'] = np.log1p(df['SalePrice'])

# Visualize the correlation
plt.figure(figsize=(10, 6))
sns.regplot(data=df, x='MORTGAGE30US', y='SalePrice', scatter_kws={'alpha':0.2}, line_kws={'color':'red'})
plt.title('Sale Price vs Mortgage Rates')
plt.show()

# Top 10 factors influencing house prices
numeric_df = df.select_dtypes(include=[np.number])
top_corrs = numeric_df.corr()['SalePrice'].sort_values(ascending=False).head(10)
print("Top 10 Correlations:\n", top_corrs)